In [1]:
import pandas as pd
import numpy as np
import warnings 
warnings.filterwarnings('ignore')

/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [11]:
target_station = ['ST-1035','ST-454','ST-471']

In [26]:
df = pd.read_csv('../../Data/Zero/2024_data.csv')
df = df.sort_values(['시작_대여소_ID','기준_날짜','시간대'])
# df = df.iloc[:,[0,1,3,4,5,6,7,8,9,10,11,12]]
df = df.iloc[:,[i for i in range(df.shape[1]) if i != 2]]
# df

df['대여건수'] = np.where(df['시작_대여소_ID'].isin(target_station), df['전체_건수'], 0)
df['반납건수'] = np.where(df['시작_대여소_ID'].isin(target_station), 0, df['전체_건수'])

df['순유출입'] = df['반납건수'] - df['대여건수']

df_group = df.groupby(['기준_날짜','시간대','시작_대여소_ID']).agg({'대여건수' : 'sum','반납건수' : 'sum','순유출입' : 'sum'})
# df['누적순유출입'] = df.groupby('시작_대여소_ID')['순유출입'].cumsum()
df_group

# df['stock'] = df['capacity'] - df['누적순유출입']
# df['stock'] = df['stock'].clip(lower=0)

대여건수  반납건수  순유출입
기준_날짜      시간대 시작_대여소_ID                  
2024-01-01 0   ST-1029     0.0   2.0   2.0
               ST-1035     2.0   0.0  -2.0
               ST-454      9.0   0.0  -9.0
               ST-479      0.0   4.0   4.0
           1   ST-1035     2.0   0.0  -2.0
...                        ...   ...   ...
2024-12-31 22  ST-471      2.0   0.0  -2.0
               ST-478      0.0   2.0   2.0
           23  ST-1038     0.0   2.0   2.0
               ST-3169     0.0   2.0   2.0
               ST-454     10.0   0.0 -10.0

[56705 rows x 3 columns]

================

In [16]:
start_df = df[df['집계_기준'] == '출발시간']
end_df   = df[df['집계_기준'] == '도착시간']
start_cnt = (
    start_df.groupby(['기준_날짜','시간대','시작_대여소_ID'])['전체_건수']
    .sum()
    .reset_index(name='대여건수')
)

end_cnt = (
    end_df.groupby(['기준_날짜','시간대','종료_대여소_ID'])['전체_건수']
    .sum()
    .reset_index(name='반납건수')
    .rename(columns={'종료_대여소_ID':'시작_대여소_ID'})
)
flow = pd.merge(
    start_cnt,
    end_cnt,
    on=['기준_날짜','시간대','시작_대여소_ID'],
    how='outer'
).fillna(0)

In [32]:
import pandas as pd
import numpy as np

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# =========================
# 0. 기본 설정
# =========================

# 분석할 스테이션 3개
target_stations = ['ST-1035','ST-454','ST-471']   # 예시, 실제 ID로 바꿔라

# 각 스테이션의 초기 자전거 수
# 실제 재고를 모르면 임시로 넣어야 한다.
initial_bikes = {
    'ST-1035' : 10,
    'ST-454' : 10,
    'ST-471' : 10
}

# 원본 복사
data = df.copy()

# 날짜형 변환
data['기준_날짜'] = pd.to_datetime(data['기준_날짜'])

# 혹시 시간대가 문자열이면 정수로 맞추기
data['시간대'] = data['시간대'].astype(int)

# =========================
# 1. 대상 스테이션 관련 행만 남기기
#    시작 또는 종료가 target_stations에 포함된 경우만 사용
# =========================
data = data[
    data['시작_대여소_ID'].isin(target_stations) |
    data['종료_대여소_ID'].isin(target_stations)
].copy()

# =========================
# 2. "스테이션별 시간단위 순유출입" 만들기
#    시작이면 대여(유출), 종료면 반납(유입)
# =========================

# 2-1. 출발(대여) 데이터
rent_out = data[data['시작_대여소_ID'].isin(target_stations)].copy()
rent_out['station_id'] = rent_out['시작_대여소_ID']
rent_out['대여건수'] = rent_out['전체_건수']
rent_out['반납건수'] = 0

# 2-2. 도착(반납) 데이터
return_in = data[data['종료_대여소_ID'].isin(target_stations)].copy()
return_in['station_id'] = return_in['종료_대여소_ID']
return_in['대여건수'] = 0
return_in['반납건수'] = return_in['전체_건수']

# 2-3. 합치기
flow_raw = pd.concat([rent_out, return_in], axis=0, ignore_index=True)

# 순유출입 정의
# 반납 - 대여
flow_raw['순유출입'] = flow_raw['반납건수'] - flow_raw['대여건수']

# =========================
# 3. 시간별 / 스테이션별 집계
# =========================

df_group = flow_raw.groupby(
    ['기준_날짜', '시간대', 'station_id'],
    as_index=False
).agg({
    '대여건수': 'sum',
    '반납건수': 'sum',
    '순유출입': 'sum',
    '온도': 'mean',
    '습도': 'mean',
    '불쾌지수': 'mean',
    '강수량': 'mean',
    '적설량': 'mean',
    '전체_이용_분': 'sum',
    '전체_이용_거리': 'sum'
})

# =========================
# 4. 스테이션별 모든 시간대 채우기
#    없는 시간은 0으로 채워야 시계열이 끊기지 않는다
# =========================

# 전체 날짜-시간대 조합 만들기
all_dates = pd.date_range(
    start=df_group['기준_날짜'].min().normalize(),
    end=df_group['기준_날짜'].max().normalize(),
    freq='D'
)

all_hours = pd.DataFrame({'시간대': range(24)})
all_date_hour = (
    pd.MultiIndex.from_product([all_dates, range(24)], names=['기준_날짜', '시간대'])
    .to_frame(index=False)
)

# 스테이션별로 full grid 생성 후 merge
full_frames = []

for station in target_stations:
    temp_full = all_date_hour.copy()
    temp_full['station_id'] = station

    temp_station = df_group[df_group['station_id'] == station].copy()

    merged = temp_full.merge(
        temp_station,
        on=['기준_날짜', '시간대', 'station_id'],
        how='left'
    )

    # 건수/유출입/이용정보 없는 시간은 0
    fill_zero_cols = [
        '대여건수', '반납건수', '순유출입',
        '전체_이용_분', '전체_이용_거리',
        '강수량', '적설량'
    ]
    for col in fill_zero_cols:
        if col in merged.columns:
            merged[col] = merged[col].fillna(0)

    # 날씨는 보간 또는 앞값/뒤값 사용
    weather_cols = ['온도', '습도', '불쾌지수']
    for col in weather_cols:
        if col in merged.columns:
            merged[col] = merged[col].interpolate().bfill().ffill()

    full_frames.append(merged)

ts_df = pd.concat(full_frames, axis=0, ignore_index=True)

# 정렬
ts_df = ts_df.sort_values(['station_id', '기준_날짜', '시간대']).reset_index(drop=True)

# =========================
# 5. datetime 컬럼 만들기
# =========================
ts_df['datetime'] = pd.to_datetime(
    ts_df['기준_날짜'].dt.strftime('%Y-%m-%d') + ' ' + ts_df['시간대'].astype(str) + ':00:00'
)

# =========================
# 6. 타겟: 스테이션별 현재 자전거 수(bike_count) 만들기
#    bike_count = 초기자전거수 + 순유출입 누적합
# =========================
print(ts_df.columns.tolist())
def make_bike_count(group):
    station = group['station_id'].iloc[0]
    init_val = initial_bikes.get(station, 0)

    group = group.sort_values('datetime').copy()
    group['bike_count'] = init_val + group['순유출입'].cumsum()

    # 음수 방지
    group['bike_count'] = group['bike_count'].clip(lower=0)
    return group

ts_df = ts_df.groupby('station_id', group_keys=False).apply(make_bike_count)
ts_df = ts_df.reset_index(drop=True)

# # =========================
# # 7. 시간 파생변수 생성
# # =========================
# ts_df['year'] = ts_df['datetime'].dt.year
# ts_df['month'] = ts_df['datetime'].dt.month
# ts_df['day'] = ts_df['datetime'].dt.day
# ts_df['dayofweek'] = ts_df['datetime'].dt.dayofweek
# ts_df['weekofyear'] = ts_df['datetime'].dt.isocalendar().week.astype(int)
# ts_df['is_weekend'] = (ts_df['dayofweek'] >= 5).astype(int)

# # cyclic encoding
# ts_df['hour_sin'] = np.sin(2 * np.pi * ts_df['시간대'] / 24)
# ts_df['hour_cos'] = np.cos(2 * np.pi * ts_df['시간대'] / 24)
# ts_df['dow_sin'] = np.sin(2 * np.pi * ts_df['dayofweek'] / 7)
# ts_df['dow_cos'] = np.cos(2 * np.pi * ts_df['dayofweek'] / 7)

# # =========================
# # 8. lag / rolling feature 생성
# #    핵심
# # =========================

# def add_lag_features(group):
#     group = group.sort_values('datetime').copy()

#     # bike_count 기준 lag
#     group['lag_1'] = group['bike_count'].shift(1)      # 1시간 전
#     group['lag_2'] = group['bike_count'].shift(2)
#     group['lag_3'] = group['bike_count'].shift(3)
#     group['lag_24'] = group['bike_count'].shift(24)    # 24시간 전
#     group['lag_48'] = group['bike_count'].shift(48)
#     group['lag_168'] = group['bike_count'].shift(168)  # 7일 전 같은 시간

#     # 순유출입 rolling
#     group['flow_mean_3'] = group['순유출입'].shift(1).rolling(3).mean()
#     group['flow_mean_6'] = group['순유출입'].shift(1).rolling(6).mean()
#     group['flow_mean_24'] = group['순유출입'].shift(1).rolling(24).mean()

#     # bike_count rolling
#     group['bike_mean_3'] = group['bike_count'].shift(1).rolling(3).mean()
#     group['bike_mean_6'] = group['bike_count'].shift(1).rolling(6).mean()
#     group['bike_mean_24'] = group['bike_count'].shift(1).rolling(24).mean()

#     return group

# ts_df = ts_df.groupby('station_id', group_keys=False).apply(add_lag_features)
# ts_df = ts_df.reset_index(drop=True)

# # =========================
# # 9. 결측 제거
# #    lag 때문에 앞부분은 NaN 생김
# # =========================
# model_df = ts_df.dropna().copy()

# # =========================
# # 10. train / test 분리
# #     2024 -> train
# #     2025 -> test
# # =========================
# train_df = model_df[model_df['year'] == 2024].copy()
# test_df = model_df[model_df['year'] == 2025].copy()

# # =========================
# # 11. feature / target 정의
# # =========================
# feature_cols = [
#     'station_id',
#     '시간대',
#     'month',
#     'day',
#     'dayofweek',
#     'weekofyear',
#     'is_weekend',
#     'hour_sin',
#     'hour_cos',
#     'dow_sin',
#     'dow_cos',
#     '온도',
#     '습도',
#     '불쾌지수',
#     '강수량',
#     '적설량',
#     '전체_이용_분',
#     '전체_이용_거리',
#     '대여건수',
#     '반납건수',
#     '순유출입',
#     'lag_1',
#     'lag_2',
#     'lag_3',
#     'lag_24',
#     'lag_48',
#     'lag_168',
#     'flow_mean_3',
#     'flow_mean_6',
#     'flow_mean_24',
#     'bike_mean_3',
#     'bike_mean_6',
#     'bike_mean_24'
# ]

# target_col = 'bike_count'

# X_train = train_df[feature_cols]
# y_train = train_df[target_col]

# X_test = test_df[feature_cols]
# y_test = test_df[target_col]

# # =========================
# # 12. 모델 학습
# # =========================
# model = HistGradientBoostingRegressor(
#     max_iter=300,
#     learning_rate=0.05,
#     max_depth=8,
#     random_state=42
# )

# model.fit(X_train, y_train)

# # =========================
# # 13. 예측 및 평가
# # =========================
# pred = model.predict(X_test)

# mae = mean_absolute_error(y_test, pred)
# rmse = np.sqrt(mean_squared_error(y_test, pred))
# r2 = r2_score(y_test, pred)

# print("MAE :", mae)
# print("RMSE:", rmse)
# print("R2  :", r2)

# # 예측값 저장
# test_df['pred_bike_count'] = pred

# # 보기 좋게 출력
# result = test_df[['datetime', 'station_id', 'bike_count', 'pred_bike_count']].copy()
# print(result.head(20))

['기준_날짜', '시간대', 'station_id', '대여건수', '반납건수', '순유출입', '온도', '습도', '불쾌지수', '강수량', '적설량', '전체_이용_분', '전체_이용_거리', 'datetime']


KeyError: 'station_id'